# **Multi-Brand Marketing Campaign Performance Analysis (Corrected)**

**Label convention used in this notebook:** `Profit = 0`, `Loss = 1` (so in `app.py` use `"Profit" if predicted_class == 0 else "Loss"`).

**Run order:** *Runtime → Restart and run all.* Do not run cells out of order (the ROI_Flag mapping must only run once).

## **Setup**

In [5]:
# ---- Configuration ---------------------------------------------------------
# Colab: path inside Google Drive.  Local Jupyter: just put final_df.csv next to this notebook.
DATA_PATH = r"D:\PROJECTS\Anna_Project_3\Multi-Brand_Marketing_Campaign_Performance_Analysis\CSV\final_df.csv"
RUN_SVM = False   # SVM on ~150K rows takes 15+ minutes. Set True only if you need it.
# -----------------------------------------------------------------------------

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import joblib

## **Load data**

In [6]:
final_df = pd.read_csv(DATA_PATH)

# Drop leftover row-number columns (they are NOT marketing data and would leak into the models)
final_df = final_df.drop(columns=["Unnamed: 0", "index"], errors="ignore")

# Rows without these values cannot be used for training
final_df = final_df.dropna(subset=["Channel_Used", "Revenue", "ROI_Flag"]).reset_index(drop=True)

print(final_df.shape)
final_df.head()

(153252, 17)


,Campaign_ID,Campaign_Type,Target_Audience,Duration,Channel_Used,Impressions,Clicks,Leads,Conversions,Revenue,Acquisition_Cost,ROI,Language,Engagement_Score,Customer_Segment,Date,ROI_Flag
0,NY-CMP-1000,Social Media,College Students,21.0,"WhatsApp, YouTube",57804.0,6156.0,3616.0,2355.0,1370861.875,207.23,1.81,Hindi,20.98,College Students,2025-04-29,Profit
1,NY-CMP-1001,Paid Ads,Tier 2 City Customers,18.0,YouTube,91801.0,3321.0,1971.0,1357.0,1046247.000,180.83,3.26,Hindi,7.24,College Students,2025-04-06,Profit
2,NY-CMP-1002,Influencer,Youth,23.0,"WhatsApp, Google, YouTube",15536.0,2182.0,952.0,755.0,197055.000,90.60,1.88,English,25.03,College Students,2025-01-14,Profit
3,NY-CMP-1003,Email,Working Women,18.0,"YouTube, Facebook, Instagram",88114.0,8413.0,2231.0,947.0,376906.000,249.07,0.60,Hindi,13.15,College Students,2025-06-04,Profit
4,NY-CMP-1004,Paid Ads,College Students,10.0,"Facebook, Instagram",96871.0,3743.0,2060.0,1258.0,518296.000,228.60,0.80,Hindi,7.29,Tier 2 City Customers,2024-12-29,Profit


# **Exploratory Data Analysis (EDA)**

### **Analyze campaign performance across brands**

In [7]:
analysis_2 = (
    final_df.groupby("Campaign_Type")["ROI"].mean()
    .reset_index().sort_values(by="ROI", ascending=False)
)
fig = px.bar(analysis_2, x="Campaign_Type", y="ROI", color="ROI", title="Average ROI by Campaign Type")
fig.show()

### **Identify top-performing and low-performing campaigns**

In [8]:
top_campaigns = final_df.groupby("Campaign_Type")["ROI"].mean().nlargest(5).reset_index()
top_campaigns

,Campaign_Type,ROI
0,Paid Ads,2.170926
1,Social Media,2.152836
2,SEO,2.150435
3,Email,2.146171
4,Influencer,2.131785


In [9]:
low_campaigns = final_df.groupby("Campaign_Type")["ROI"].mean().nsmallest(5).reset_index()
low_campaigns

,Campaign_Type,ROI
0,Influencer,2.131785
1,Email,2.146171
2,SEO,2.150435
3,Social Media,2.152836
4,Paid Ads,2.170926


### **Explore relationships between spend, clicks, revenue, and ROI**

In [10]:
corr_matrix = final_df[["Acquisition_Cost", "Clicks", "Revenue", "ROI"]].corr()
fig = px.imshow(corr_matrix, text_auto=True, color_continuous_scale="RdBu_r",
                title="Correlation Between Spend, Clicks, Revenue, and ROI")
fig.show()

### **Analyze channel-wise effectiveness**

In [11]:
channel_eda = final_df.copy()
channel_eda["Channel_Used"] = channel_eda["Channel_Used"].str.split(", ")
channel_eda = channel_eda.explode("Channel_Used")

analysis_channel = (
    channel_eda.groupby("Channel_Used")
    .agg({"Revenue": "sum", "ROI": "mean", "Clicks": "sum",
          "Conversions": "sum", "Leads": "sum", "Impressions": "sum"})
    .reset_index().sort_values(by="ROI", ascending=False)
)
analysis_channel

,Channel_Used,Revenue,ROI,Clicks,Conversions,Leads,Impressions
0,Email,2.405592e+10,2.175943,2.380212e+08,5.036417e+07,93579045.0,2.814183e+09
3,Instagram,2.412612e+10,2.169851,2.378365e+08,5.048938e+07,93528552.0,2.817075e+09
1,Facebook,2.395987e+10,2.148465,2.373479e+08,5.027888e+07,93215914.5,2.819927e+09
2,Google,2.385267e+10,2.140691,2.355146e+08,4.981124e+07,92441340.5,2.806382e+09
4,WhatsApp,2.389286e+10,2.140504,2.365846e+08,5.013085e+07,92931951.0,2.806107e+09
5,YouTube,2.382342e+10,2.140365,2.368446e+08,5.001778e+07,92839517.5,2.818644e+09


## **Encoding**

In [12]:
from sklearn.preprocessing import LabelEncoder

label_columns = ["Campaign_Type", "Target_Audience", "Language", "Customer_Segment"]
label_encoders = {}

for col in label_columns:
    le = LabelEncoder()
    final_df[col] = le.fit_transform(final_df[col])
    label_encoders[col] = le

joblib.dump(label_encoders, "label_encoders.pkl")
print("Label encoders saved.")

Label encoders saved.


In [13]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
channel_encoded = mlb.fit_transform(final_df["Channel_Used"].str.split(", "))

# "Channel_" prefix must match what app.py builds
channel_df = pd.DataFrame(
    channel_encoded,
    columns=[f"Channel_{c}" for c in mlb.classes_],
    index=final_df.index,
)
final_df = pd.concat([final_df.drop(columns=["Channel_Used"]), channel_df], axis=1)
print("Channel columns:", list(channel_df.columns))

Channel columns: ['Channel_Email', 'Channel_Facebook', 'Channel_Google', 'Channel_Instagram', 'Channel_WhatsApp', 'Channel_YouTube']


In [14]:
print(final_df["ROI_Flag"].value_counts())

# Profit = 0, Loss = 1  (safe to re-run: only maps if the column still holds text)
if not pd.api.types.is_numeric_dtype(final_df["ROI_Flag"]):
    final_df["ROI_Flag"] = final_df["ROI_Flag"].map({"Profit": 0, "Loss": 1})

print(final_df["ROI_Flag"].value_counts(normalize=True))

ROI_Flag
Profit    119493
Loss       33759
Name: count, dtype: int64
ROI_Flag
0    0.779716
1    0.220284
Name: proportion, dtype: float64


## **Feature Engineering**

`Acquisition_Cost` is cost **per conversion**, so total cost = `Acquisition_Cost × Conversions`. The funnel ratios carry real signal without leaking the target. Divisions by zero are handled.

In [15]:
# final_df["Total_Acquisition_Cost"] = final_df["Acquisition_Cost"] * final_df["Conversions"]
# final_df["CTR"] = (final_df["Clicks"] / final_df["Impressions"].replace(0, np.nan)).fillna(0)
# final_df["Lead_Rate"] = (final_df["Leads"] / final_df["Clicks"].replace(0, np.nan)).fillna(0)
# final_df["Conversion_Rate"] = (final_df["Conversions"] / final_df["Leads"].replace(0, np.nan)).fillna(0)

# final_df[["Total_Acquisition_Cost", "CTR", "Lead_Rate", "Conversion_Rate"]].describe()

# **Model Building**

## **Regression Model, predicting Revenue**

`ROI`, `ROI_Flag`, `Campaign_ID` and `Date` are excluded (ROI is derived from Revenue, so keeping it would be leakage). Expect R² of about 0.75; revenue has a lot of built-in randomness.

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

In [17]:
X_r = final_df.drop(columns=["Campaign_ID", "ROI_Flag", "Revenue", "ROI", "Date"])
y_r = final_df["Revenue"]

print(len(X_r.columns), "regression features:")
print(list(X_r.columns))
assert "Unnamed: 0" not in X_r.columns and "index" not in X_r.columns

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_r, y_r, test_size=0.20, random_state=42)

scaler_r = StandardScaler()
X_train_rs = scaler_r.fit_transform(X_train_r)
X_test_rs = scaler_r.transform(X_test_r)

17 regression features:
['Campaign_Type', 'Target_Audience', 'Duration', 'Impressions', 'Clicks', 'Leads', 'Conversions', 'Acquisition_Cost', 'Language', 'Engagement_Score', 'Customer_Segment', 'Channel_Email', 'Channel_Facebook', 'Channel_Google', 'Channel_Instagram', 'Channel_WhatsApp', 'Channel_YouTube']


In [18]:
X_r

,Campaign_Type,Target_Audience,Duration,Impressions,Clicks,Leads,Conversions,Acquisition_Cost,Language,Engagement_Score,Customer_Segment,Channel_Email,Channel_Facebook,Channel_Google,Channel_Instagram,Channel_WhatsApp,Channel_YouTube
0,4,0,21.000000,57804.0,6156.0,3616.0,2355.0,207.23,2,20.98,0,0,0,0,0,1,1
1,2,2,18.000000,91801.0,3321.0,1971.0,1357.0,180.83,2,7.24,0,0,0,0,0,0,1
2,1,4,23.000000,15536.0,2182.0,952.0,755.0,90.60,1,25.03,0,0,0,1,0,1,1
3,0,3,18.000000,88114.0,8413.0,2231.0,947.0,249.07,2,13.15,0,0,1,0,1,0,1
4,2,0,10.000000,96871.0,3743.0,2060.0,1258.0,228.60,2,7.29,2,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153247,1,1,15.000000,25929.0,1388.0,808.0,567.0,258.43,2,10.66,3,0,0,1,0,1,1
153248,2,1,26.000000,33252.0,2871.0,950.0,777.0,421.81,1,12.96,2,0,1,0,0,0,0
153249,3,3,20.000000,54886.0,1578.0,634.0,777.0,436.24,1,4.66,3,0,0,1,1,0,0
153250,0,1,13.000000,97954.0,11480.0,4567.0,2144.0,78.77,2,18.57,1,0,1,0,1,0,0


In [19]:
def evaluate_regressor(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    metrics = {
        "Model": name,
        "r2 Score": r2_score(y_test, preds),
        "MAE": mean_absolute_error(y_test, preds),
        "MSE": mse,
        "RMSE": float(np.sqrt(mse)),
    }
    print(f"--- {name} ---")
    print(f"r2 Score : {metrics['r2 Score']:.4f} | MAE: {metrics['MAE']:,.0f} | RMSE: {metrics['RMSE']:,.0f}\n")
    return model, preds, metrics

In [20]:
reg_models = {
    "Linear Regression": LinearRegression(),
    "KNN": KNeighborsRegressor(n_neighbors=3),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}
if HAS_XGB:
    reg_models["XGBoost"] = XGBRegressor(objective="reg:squarederror", random_state=42)

reg_results = []
reg_fitted = {}

for name, model in reg_models.items():
    fitted, preds, metrics = evaluate_regressor(name, model, X_train_rs, y_train_r, X_test_rs, y_test_r)
    reg_results.append(metrics)
    reg_fitted[name] = fitted

--- Linear Regression ---
r2 Score : 0.7148 | MAE: 141,035 | RMSE: 193,739

--- KNN ---
r2 Score : 0.6099 | MAE: 165,782 | RMSE: 226,616

--- Decision Tree ---
r2 Score : 0.4810 | MAE: 179,902 | RMSE: 261,372

--- Random Forest ---
r2 Score : 0.7390 | MAE: 136,053 | RMSE: 185,341

--- Gradient Boosting ---
r2 Score : 0.7447 | MAE: 134,492 | RMSE: 183,315

--- XGBoost ---
r2 Score : 0.7388 | MAE: 134,916 | RMSE: 185,439



In [21]:
reg_results_df = pd.DataFrame(reg_results)
best_reg_name = reg_results_df.loc[reg_results_df["r2 Score"].idxmax(), "Model"]
best_reg_model = reg_fitted[best_reg_name]

print("Best regression model:", best_reg_name)
print("Best r2 Score:", round(reg_results_df["r2 Score"].max(), 4))
reg_results_df.sort_values("r2 Score", ascending=False)

Best regression model: Gradient Boosting
Best r2 Score: 0.7447


,Model,r2 Score,MAE,MSE,RMSE
4,Gradient Boosting,0.744703,134491.887943,3.360444e+10,183315.141339
3,Random Forest,0.739030,136052.852237,3.435121e+10,185340.797317
5,XGBoost,0.738754,134915.902058,3.438755e+10,185438.815294
0,Linear Regression,0.714843,141035.170976,3.753498e+10,193739.470510
1,KNN,0.609853,165782.072362,5.135465e+10,226615.637169
2,Decision Tree,0.481002,179901.817045,6.831516e+10,261371.696917


In [22]:
joblib.dump(best_reg_model, "best_Regression_model.pkl")
joblib.dump(scaler_r, "Regression_scaler.pkl")
joblib.dump(X_r.columns.tolist(), "model_columns_regression.pkl")
print("Regression files saved (model, scaler, columns).")

Regression files saved (model, scaler, columns).


## **Classification Model, predicting ROI_Flag (Profit / Loss)**

Target is imbalanced (~78% Profit / 22% Loss), so we stratify the split, use `class_weight="balanced"` where supported, and judge models by **F1 on the Loss class** (`Loss = 1`, sklearn's default positive class).

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

try:
    from xgboost import XGBClassifier
    HAS_XGB_C = True
except ImportError:
    HAS_XGB_C = False

In [24]:
X_c = final_df.drop(columns=["Campaign_ID", "Revenue", "ROI", "ROI_Flag", "Date"])
y_c = final_df["ROI_Flag"]

print(len(X_c.columns), "classification features")
print(y_c.value_counts(normalize=True).round(3).to_dict(), "(0 = Profit, 1 = Loss)")
assert list(X_c.columns) == list(X_r.columns), "Regression and classification features differ!"

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_c, y_c, test_size=0.20, random_state=42, stratify=y_c
)

scaler_c = StandardScaler()
X_train_cs = scaler_c.fit_transform(X_train_c)
X_test_cs = scaler_c.transform(X_test_c)

17 classification features
{0: 0.78, 1: 0.22} (0 = Profit, 1 = Loss)


In [25]:
def evaluate_classifier(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)
    result = {
        "Model": name,
        "Accuracy_Train": accuracy_score(y_train, train_preds),
        "Accuracy_Test": accuracy_score(y_test, test_preds),
        "Precision_Test": precision_score(y_test, test_preds, zero_division=0),
        "Recall_Test": recall_score(y_test, test_preds, zero_division=0),
        "F1_Train": f1_score(y_train, train_preds, zero_division=0),
        "F1_Test": f1_score(y_test, test_preds, zero_division=0),
    }
    result["Accuracy_Gap"] = result["Accuracy_Train"] - result["Accuracy_Test"]
    result["F1_Gap"] = result["F1_Train"] - result["F1_Test"]
    print(f"--- {name} ---")
    print(f"Test Accuracy: {result['Accuracy_Test']:.4f} | Test F1 (Loss): {result['F1_Test']:.4f} | "
          f"Train/Test Acc Gap: {result['Accuracy_Gap']:.4f}\n")
    return model, test_preds, result

In [26]:
clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight="balanced", n_jobs=-1),
    "Naive Bayes": GaussianNB(),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
}
if RUN_SVM:
    clf_models["SVM"] = SVC(class_weight="balanced")
if HAS_XGB_C:
    clf_models["XGBoost"] = XGBClassifier(eval_metric="logloss", random_state=42)

# Outside any if-block, so they always exist
clf_results = []
clf_fitted = {}

for name, model in clf_models.items():
    fitted, preds, result = evaluate_classifier(name, model, X_train_cs, y_train_c, X_test_cs, y_test_c)
    clf_results.append(result)
    clf_fitted[name] = fitted

--- Logistic Regression ---
Test Accuracy: 0.8939 | Test F1 (Loss): 0.7835 | Train/Test Acc Gap: -0.0035

--- KNN ---
Test Accuracy: 0.8873 | Test F1 (Loss): 0.7267 | Train/Test Acc Gap: 0.0449

--- Decision Tree ---
Test Accuracy: 0.8712 | Test F1 (Loss): 0.7050 | Train/Test Acc Gap: 0.1288

--- Random Forest ---
Test Accuracy: 0.9035 | Test F1 (Loss): 0.7908 | Train/Test Acc Gap: 0.0930

--- Naive Bayes ---
Test Accuracy: 0.8311 | Test F1 (Loss): 0.7007 | Train/Test Acc Gap: -0.0030

--- Gradient Boosting ---
Test Accuracy: 0.9116 | Test F1 (Loss): 0.7875 | Train/Test Acc Gap: -0.0005

--- AdaBoost ---
Test Accuracy: 0.9079 | Test F1 (Loss): 0.7878 | Train/Test Acc Gap: -0.0018

--- XGBoost ---
Test Accuracy: 0.9087 | Test F1 (Loss): 0.7805 | Train/Test Acc Gap: 0.0146



In [27]:
clf_results_df = pd.DataFrame(clf_results)
clf_results_df[["Model", "Accuracy_Train", "Accuracy_Test", "Accuracy_Gap",
                "Precision_Test", "Recall_Test", "F1_Test"]].sort_values("F1_Test", ascending=False)

,Model,Accuracy_Train,Accuracy_Test,Accuracy_Gap,Precision_Test,Recall_Test,F1_Test
3,Random Forest,0.996419,0.903462,0.092958,0.756526,0.828347,0.790809
6,AdaBoost,0.906118,0.907899,-0.001780,0.799786,0.776214,0.787824
5,Gradient Boosting,0.911020,0.911553,-0.000532,0.836358,0.744076,0.787523
0,Logistic Regression,0.890376,0.893902,-0.003526,0.711608,0.871594,0.783518
7,XGBoost,0.923271,0.908714,0.014557,0.829830,0.736671,0.780480
1,KNN,0.932252,0.887345,0.044907,0.780480,0.679799,0.726668
2,Decision Tree,1.000000,0.871195,0.128805,0.711399,0.698756,0.705021
4,Naive Bayes,0.828044,0.831066,-0.003022,0.574597,0.897808,0.700728


### **Cross-validation** (checks the score is stable and not a lucky split)

In [28]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    GradientBoostingClassifier(random_state=42),
    X_train_cs, y_train_c, cv=cv, scoring="f1", n_jobs=-1,
)
print("CV F1 scores:", scores.round(4))
print("Mean CV F1:", round(scores.mean(), 4))
print("Std CV F1:", round(scores.std(), 4))

CV F1 scores: [0.7814 0.7862 0.7837 0.7836 0.7833]
Mean CV F1: 0.7837
Std CV F1: 0.0015


In [29]:
best_clf_name = clf_results_df.loc[clf_results_df["F1_Test"].idxmax(), "Model"]
best_clf_model = clf_fitted[best_clf_name]

print("Best Classifier:", best_clf_name)
print("Best Test Accuracy:", round(clf_results_df.loc[clf_results_df["Model"] == best_clf_name, "Accuracy_Test"].values[0], 4))
print("Best Test F1 (Loss):", round(clf_results_df["F1_Test"].max(), 4))

Best Classifier: Random Forest
Best Test Accuracy: 0.9035
Best Test F1 (Loss): 0.7908


In [31]:
joblib.dump(best_clf_model, "best_classification_model.pkl")
joblib.dump(scaler_c, "classification_scaler.pkl")
joblib.dump(X_c.columns.tolist(), "model_columns_class.pkl")
print("Classification files saved (model, scaler, columns).")
print("\nAll 7 files ready: best_classification_model, classification_scaler, model_columns_class,")
print("best_Regression_model, Regression_scaler, model_columns_regression, label_encoders (.pkl)")

Classification files saved (model, scaler, columns).

All 7 files ready: best_classification_model, classification_scaler, model_columns_class,
best_Regression_model, Regression_scaler, model_columns_regression, label_encoders (.pkl)


## **Sanity check: 8 real campaigns with known results**

These come from the held-out test split. Expect at least **7 of 8** labels correct. Revenue is an estimate (typical error about ₹135K).

In [32]:
PROFIT_CLASS = 0   # this notebook maps Profit = 0, Loss = 1
CHANNELS = ["WhatsApp", "YouTube", "Google", "Facebook", "Instagram", "Email"]

clf = joblib.load("best_classification_model.pkl")
clf_scaler = joblib.load("classification_scaler.pkl")
clf_cols = joblib.load("model_columns_class.pkl")
reg = joblib.load("best_Regression_model.pkl")
reg_scaler = joblib.load("Regression_scaler.pkl")
reg_cols = joblib.load("model_columns_regression.pkl")
encoders = joblib.load("label_encoders.pkl")

print("Classifier features:", len(clf_cols), "| junk cols:", [c for c in clf_cols if c in ("Unnamed: 0", "index")])

# (type, audience, duration, channels, impressions, clicks, leads, conversions, acq_cost, language, engagement, segment, actual_revenue, actual_label)
samples = [
    ("SEO", "Premium Shoppers", 15, ["Email"], 90648, 5309, 2712, 1769, 168.01, "Hindi", 10.8, "College Students", 1273680, "Profit"),
    ("Paid Ads", "Working Women", 14, ["YouTube", "Facebook"], 55820, 5087, 2937, 1280, 66.28, "Tamil", 16.67, "College Students", 359832, "Profit"),
    ("Paid Ads", "Youth", 23, ["WhatsApp", "Google", "YouTube"], 76381, 8599, 4847, 1621, 170.21, "English", 19.73, "Working Women", 1071481, "Profit"),
    ("Email", "Premium Shoppers", 5, ["YouTube"], 67277, 7652, 3436, 777, 176.84, "Hindi", 18.98, "Tier 2 City Customers", 979952, "Profit"),
    ("Social Media", "Youth", 30, ["Google", "YouTube"], 42242, 1050, 217, 70, 850.34, "Hindi", 3.17, "Working Women", 14420, "Loss"),
    ("Social Media", "Working Women", 8, ["Instagram"], 15390, 633, 352, 264, 862.96, "Hindi", 8.12, "Premium Shoppers", 119328, "Loss"),
    ("Influencer", "Premium Shoppers", 30, ["Instagram"], 11518, 828, 224, 775, 862.96, "Bengali", 10.19, "Working Women", 28426, "Loss"),
    ("Social Media", "College Students", 25, ["YouTube", "Email"], 15328, 1661, 623, 231, 862.96, "Hindi", 16.41, "Premium Shoppers", 106029, "Loss"),
]

rows = []
for (ctype, aud, dur, chans, imp, clk, lds, conv, cost, lang, eng, seg, _, _) in samples:
    d = {"Campaign_Type": ctype, "Target_Audience": aud, "Duration": dur, "Impressions": imp,
         "Clicks": clk, "Leads": lds, "Conversions": conv, "Acquisition_Cost": cost,
         "Language": lang, "Engagement_Score": eng, "Customer_Segment": seg}
    for ch in CHANNELS:
        d[f"Channel_{ch}"] = int(ch in chans)
    rows.append(d)

t = pd.DataFrame(rows)
t["Total_Acquisition_Cost"] = t["Acquisition_Cost"] * t["Conversions"]
t["CTR"] = t["Clicks"] / t["Impressions"]
t["Lead_Rate"] = t["Leads"] / t["Clicks"]
t["Conversion_Rate"] = t["Conversions"] / t["Leads"]
for col in ["Campaign_Type", "Target_Audience", "Language", "Customer_Segment"]:
    t[col] = encoders[col].transform(t[col])

pred_rev = reg.predict(reg_scaler.transform(t.reindex(columns=reg_cols, fill_value=0)))
pred_cls = clf.predict(clf_scaler.transform(t.reindex(columns=clf_cols, fill_value=0)))

out = pd.DataFrame({
    "Sample": [f"P{i}" for i in range(1, 5)] + [f"L{i}" for i in range(1, 5)],
    "Actual": [s[13] for s in samples],
    "Predicted": ["Profit" if c == PROFIT_CLASS else "Loss" for c in pred_cls],
    "Actual_Rev": [s[12] for s in samples],
    "Pred_Rev": pred_rev.round(0),
})
out["Rev_Error"] = (out["Pred_Rev"] - out["Actual_Rev"]).round(0)
out["Label_OK"] = out["Actual"] == out["Predicted"]

print(out.to_string(index=False))
print(f"\nLabel accuracy: {out['Label_OK'].sum()}/8")
print(f"Mean abs revenue error: {out['Rev_Error'].abs().mean():,.0f}  (typical ~135,000-175,000)")

Classifier features: 17 | junk cols: []
Sample Actual Predicted  Actual_Rev  Pred_Rev  Rev_Error  Label_OK
    P1 Profit    Profit     1273680  841972.0  -431708.0      True
    P2 Profit    Profit      359832  638280.0   278448.0      True
    P3 Profit    Profit     1071481  800385.0  -271096.0      True
    P4 Profit    Profit      979952  744517.0  -235435.0      True
    L1   Loss      Loss       14420   57026.0    42606.0      True
    L2   Loss      Loss      119328  128442.0     9114.0      True
    L3   Loss      Loss       28426   88913.0    60487.0      True
    L4   Loss      Loss      106029  131017.0    24988.0      True

Label accuracy: 8/8
Mean abs revenue error: 169,235  (typical ~135,000-175,000)
